# Session 2 — Object Selection and Event Reconstruction

In this session we turn the physics picture from Session 1 into **concrete object selections** for the bb + MET search: good jets, b-tagged jets, isolated leptons for the veto, and basic event cleaning. These are the same ingredients used in the full CMS bb + MET analysis.

**Learning objectives:**
- Understand physics object selection in CMS in the context of a bb + MET search
- Apply jet quality and kinematic cuts consistent with Run-2 analyses
- Use b-tagging (DeepJet/DeepFlav discriminator) to identify b-jets
- Implement a lepton veto as used in bbMET signal regions
- Build event-level selection logic that will later define the bbMET signal region

<img src="figures/session2_fig_PFT-25-001_preview.png" width="100%" style="border-radius:8px;">



In [1]:
from config.cms_display import show_cms_event
show_cms_event("PFT-25-001", "session2_fig_PFT-25-001_preview.png")

---
## 1. Physics Objects in CMS

<div style="display: flex; gap: 0.5rem; align-items: flex-start;">
  <div style="flex: 1; min-width: 0;">

Reconstructed objects (jets, electrons, muons, MET) are built from detector signals. We apply **identification** and **selection** criteria to reduce fake rates and align with analysis definitions.

For the bbDM search we need:
- **Good jets** (quality + kinematic cuts)
- **b-tagged jets** (heavy-flavor tagging)
- **No tight leptons** (lepton veto)

  </div>
  <div style="flex: 0 0 40%; max-width: 50%; text-align: left;">
    <img src="figures/session2_fig_top_quark_decay.png" alt="Top quark decay schematic" style="width: 100%; height: auto; display: block;"/>
  </div>
</div>

---
## 2. Jet Selection Criteria

Typical jet selection for Run-2 analyses:
- **pT > 30 GeV** (above noise and trigger thresholds)
- **|η| < 2.4** (tracker coverage for b-tagging)
- **Jet ID** (quality flags; in NanoAOD often `jetId >= 2` for tight)

We will implement this in Coffea using a **mask** on the jet array.

In [2]:
import sys
sys.path.append("..")
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

def load_events(filepath):
    """Load one NanoAOD ROOT file and return a NanoEvents object.

    - **Why NanoEvents?** It maps flat NanoAOD branches like `Jet_pt`, `MET_pt` into
      convenient collections like `events.Jet.pt`, `events.MET.pt` using Awkward Arrays.
    """
    return NanoEventsFactory.from_root(filepath, schemaclass=NanoAODSchema).events()

# Load events
from config.datasets_2017 import get_one_file_per_group_from_yaml
try:
    files = get_one_file_per_group_from_yaml()
    # We load one *data* file (MET run) and one *MC background* file.
    events = load_events(files["background"][0])
    print("Files Loaded:\n",files)
except Exception as e:
    print("Could not load events.")
    print(e)


/cvmfs/sft.cern.ch/lcg/views/LCG_105a_swan/x86_64-el9-gcc13-opt/lib/python3.9/site-packages/coffea/nanoevents/schemas/nanoaod.py:201: RuntimeWarning: Missing cross-reference index for FatJet_genJetAK8Idx => GenJetAK8
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_105a_swan/x86_64-el9-gcc13-opt/lib/python3.9/site-packages/coffea/nanoevents/schemas/nanoaod.py:201: RuntimeWarning: Missing cross-reference index for FsrPhoton_muonIdx => Muon
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_105a_swan/x86_64-el9-gcc13-opt/lib/python3.9/site-packages/coffea/nanoevents/schemas/nanoaod.py:201: RuntimeWarning: Missing cross-reference index for Muon_fsrPhotonIdx => FsrPhoton
  warnings.warn(


Files Loaded:
 {'data': ['/eos/cms/store/group/phys_susy/sus-23-008/cmsdas2026/2017/SingleElectron-Run2017B-02Apr2020-v1/0ACA0341-E365-2544-99F0-FBA4C92C0301.root'], 'background': ['/eos/cms/store/group/phys_susy/sus-23-008/cmsdas2026/2017/DYJetsToLL_M-50_HT-400to600_TuneCP5_13TeV-madgraphMLM-pythia8-RunIIFall17NanoAOD-PU2017_12Apr2018_94X_mc2017_realistic_v14-v1/2422C97F-2B42-E811-A0A4-0CC47A4D7694.root']}


In [3]:
import numpy as np
import awkward as ak

JET_PT_MIN = 30.0   # GeV
JET_ETA_MAX = 2.5

def select_good_jets(events):
    jets = events.Jet
    mask = (
        (jets.pt > JET_PT_MIN) &
        (np.abs(jets.eta) < JET_ETA_MAX) &
        (jets.jetId >= 2)
    )
    return jets[mask]

print("Function select_good_jets defined. Call it with your events.")

Function select_good_jets defined. Call it with your events.


### Exercise 2.1
Load your events (as in Session 1) and apply `select_good_jets`. Compare the number of jets **before** and **after** selection (e.g. total jet count and jet multiplicity per event).

In [4]:
# Your code: load events, get good_jets, compare multiplicities
good_jets = select_good_jets(events)
njets_before = ak.num(events.Jet)
njets_after = ak.num(good_jets)
print("Mean jets/event before:", ak.mean(njets_before))
print("Mean jets/event after:", ak.mean(njets_after))

Mean jets/event before: 8.472841995992884
Mean jets/event after: 4.173759762124168


---
## 3. Introduction to b-Tagging

**b-tagging** identifies jets likely from b quarks using secondary vertices and track impact parameters. CMS provides discriminants from multivariate algorithms.

We use **DeepJet** (or DeepFlav) and the branch **btagDeepFlavB**. A working point (WP) is a threshold on this discriminant; higher threshold = purer b-jets, lower efficiency.

<p align="center">
  <img src="figures/session2_fig_jets_schematics.png" alt="Jets schematic" style="width: 100%; max-width: 720px; height: auto; margin-top: 0.75rem;"/>
</p>

---
## 4. DeepJet Discriminator

**Medium working point** (example for 2017):
`btagDeepFlavB > 0.2783`

Count b-jets per event by summing jets that pass this cut.

In [7]:
BTAG_WP_MEDIUM = 0.2783  # DeepFlavB medium (2017); adjust for your NanoAOD year


def count_bjets(jets, wp=BTAG_WP_MEDIUM):
    return ak.sum(jets.btagDeepFlavB > wp, axis=1)

# After getting good_jets:
nbjets = count_bjets(good_jets)
print("Example b-jet counts (first 10 events):", nbjets[:10])

Example b-jet counts (first 10 events): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### Exercise 2.2
Using `good_jets` and `count_bjets`, plot the **b-jet multiplicity** (0, 1, 2, 3, ...). Use a histogram with bins 0–6. Label axes.

In [ ]:
import matplotlib.pyplot as plt

# Your code: b-jet multiplicity histogram
# nbjets = count_bjets(good_jets)
# plt.hist(ak.to_numpy(nbjets), bins=range(7), ...)

---
## 5. Lepton Identification

For the **lepton veto** we need to define **tight** electrons and muons. Events with one or more tight leptons are rejected.

Typical requirements:
- **Electrons:** pT > 10 GeV, |η| < 2.5, cutBased ID (e.g. tight = 4 in older NanoAOD, or >= 2)
- **Muons:** pT > 10 GeV, |η| < 2.4, tightId, isolation (e.g. pfRelIso04_all < 0.15)

In [ ]:
LEP_PT_MIN = 10.0
LEP_ETA_MAX_EL = 2.5
LEP_ETA_MAX_MU = 2.4

def select_tight_electrons(events):
    ele = events.Electron
    mask = (
        (ele.pt > LEP_PT_MIN) &
        (np.abs(ele.eta) < LEP_ETA_MAX_EL) &
        (ele.cutBased >= 2)
    )
    return ele[mask]

def select_tight_muons(events):
    mu = events.Muon
    mask = (
        (mu.pt > LEP_PT_MIN) &
        (np.abs(mu.eta) < LEP_ETA_MAX_MU) &
        (mu.tightId) &
        (mu.pfRelIso04_all < 0.15)
    )
    return mu[mask]

def n_tight_leptons(events):
    nele = ak.count(select_tight_electrons(events).pt, axis=1)
    nmu = ak.count(select_tight_muons(events).pt, axis=1)
    return nele + nmu

print("Lepton selection functions defined.")

### Exercise 2.3
Compute `n_tight_leptons(events)` and plot the distribution (0, 1, 2, ...). How many events have exactly 0 tight leptons?

In [ ]:
# Your code: n_tight_leptons and plot
# nlep = n_tight_leptons(events)
# plt.hist(ak.to_numpy(nlep), bins=range(6), ...)
# print("Events with 0 tight leptons:", ak.sum(nlep == 0))

---
## 6. Event Cleaning

In full analyses we also apply **event filters** (e.g. bad muons, duplicate events). For this exercise we keep the selection as: good jets, b-tag count, and lepton veto.

**Event selection so far:**
- At least one good jet (for plots)
- Optionally: ≥2 b-jets and 0 leptons for a pre-signal region

---
## 7. Putting It Together

Build a mask for events that pass:
- ≥1 good jet
- (Optional) ≥2 b-jets, 0 tight leptons

Then plot **jet multiplicity**, **b-jet multiplicity**, and **leading jet pT** after this selection.

In [ ]:
# Example: event mask with good jets only
# good_jets = select_good_jets(events)
# njets = ak.num(good_jets)
# nbjets = count_bjets(good_jets)
# nlep = n_tight_leptons(events)
# mask = (njets >= 1)
# good_events = events[mask]
# good_jets_sel = good_jets[mask]
# leading_pt = ak.fill_none(ak.firsts(good_jets_sel.pt), 0.0)

### Exercise 2.4 — Jet multiplicity after selection
Plot **jet multiplicity** (using `ak.num(good_jets)`) for events with at least one good jet. Use bins 0–15.

In [ ]:
# Your code: jet multiplicity after selection

### Exercise 2.5 — b-jet multiplicity
Plot **b-jet multiplicity** (same as Exercise 2.2; you can overlay or plot separately).

In [ ]:
# Your code: b-jet multiplicity

### Exercise 2.6 — Leading jet pT
Plot the **leading (highest pT) jet pT** per event. Use `ak.firsts(good_jets.pt)` (with `ak.fill_none(..., 0)` if needed). Range 0–500 GeV.

In [ ]:
# Your code: leading jet pT
# lead_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
# plt.hist(ak.to_numpy(lead_pt), bins=50, range=(0, 500), ...)

---
## 8. Modify Thresholds (Discussion)

**Try changing:**
- Jet pT cut from 30 to 40 GeV. How do jet and b-jet multiplicities change?
- b-tag WP: use a looser value (e.g. 0.2) or tighter (e.g. 0.5). How does b-jet multiplicity change?

Discuss with your neighbour: why does tt̄ dominate when we require ≥2 b-jets?

In [ ]:
# Optional: re-run with JET_PT_MIN = 40 and compare histograms

---
## 9. Summary — Session 2

- **Good jets:** pT > 30 GeV, |η| < 2.4, jetId >= 2.
- **b-jets:** btagDeepFlavB > 0.2783 (medium WP).
- **Lepton veto:** 0 tight electrons and 0 tight muons.
- We built event-level quantities: njets, nbjets, nlep, leading jet pT.

**Next session:** We will define the signal region (MET > 200 GeV, ≥2 b-jets, 0 leptons), compare yields, and introduce control regions.